# Matched generation
Attach the saved outputs of the training notebook. Run in the Qwen environment on T4 x2. The code finds completed SFT/HH/UF runs by their saved metadata, never by guessing the latest checkpoint. Inspect the three sanity responses produced by each generator. Final decoding is greedy, 256 new tokens, repetition penalty 1.1, n-gram block 0.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json
ROOT=Path('/kaggle/working/cs5326-pa2-small-model-alignment')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/salehfarooq/cs5326-pa2-small-model-alignment.git',str(ROOT)],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH']=str(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-qwen.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.','--no-deps'],check=True)


In [ ]:
runs={}
for p in Path('/kaggle/input').rglob('run.json'):
    try:r=json.loads(p.read_text())
    except (ValueError,OSError):continue
    track=r.get('metadata',{}).get('track')
    if track in ['sft','hh','uf'] and r.get('status')=='completed' and (p.parent/'best/state.json').exists():
        if track in runs:raise ValueError('Multiple training runs for '+track+'; attach only the intended saved version.')
        runs[track]=p.parent
assert set(runs)=={'sft','hh','uf'}
OUT=Path('/kaggle/working/matched_evaluation_reproduction');OUT.mkdir(exist_ok=False)
for track in ['sft','hh','uf']:
    for bank,bankfile in [('alpaca_eval',ROOT/'data/general/alpaca_eval/eval.jsonl'),('hh',ROOT/'eval_sets/hh_rlhf_eval_300_seed42.json')]:
        subprocess.run([sys.executable,'-u','-m','scripts.generate_checkpoint',
           '--checkpoint',str(runs[track]/'best'),'--bank',str(bankfile),'--bank-kind',bank,
           '--generator',{'sft':'qwen2.5-0.5b-sft','hh':'qwen2.5-0.5b-dpo-hh','uf':'qwen2.5-0.5b-dpo-uf'}[track],'--output-dir',str(OUT/(track+'-'+bank)),
           '--repetition-penalty','1.1','--no-repeat-ngram-size','0','--max-new-tokens','256'],check=True)
for track in ['hh','uf']:
    for bank in ['alpaca_eval','hh']:
        subprocess.run([sys.executable,'-m','scripts.build_pairs','--sft-dir',str(OUT/('sft-'+bank)),
           '--dpo-dir',str(OUT/(track+'-'+bank)),
           '--output-dir',str(OUT/('pairs-'+track+'-'+bank)),'--seed','42'],check=True)
